# 🇧🇩 "Amar Tax" AI Agent - Virtual Tax Advisor
### Applied AI with Multi-Agent Systems (CrewAI) | Intermediate Level

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/drive/1UfgNv61I-ICv6S-iWflV8yC0FJtt1HLq)
[![GitHub Repo](https://img.shields.io/badge/GitHub-Repository-blue?logo=github)](https://github.com/Udipto-Mondal/Amar-Tax-AI-Agent.git)

---

## 📌 Project Overview
Filing an individual income tax return in Bangladesh confuses many people: who must file, what the tax-free threshold is, how the investment rebate is applied, and which documents are required.

This project builds an autonomous **Multi-Agent System (MAS)** using **CrewAI** that acts as a **"Virtual Tax Advisor"**.
The system takes a taxpayer profile (**Age, Gender, Profession, Income, Investment, Location, TIN status**) and outputs a comprehensive **Tax Readiness Report** featuring:
1. **English Markdown Summary Table**
2. **Audit / Edge Case Warning Flags**
3. **3–4 line Fluent Bangla Summary**

---

## 👥 The Crew Definition (4 Specialized Agents)
| # | Agent Role | Goal & Responsibilities | Tools Used |
|---|---|---|---|
| **1** | **Senior Taxpayer Classification Officer** | Determines mandatory filing status, taxpayer category, tax-free threshold, and regional minimum tax. Flags minor status. | `lookup_tax_db` |
| **2** | **Chartered Tax Accountant** | Calculates slab-based gross tax, allowable investment cap (20%), 15% investment rebate, and net payable tax vs minimum tax. | `lookup_tax_db`, `calculate_slab_tax` |
| **3** | **Return Filing Documentation Officer** | Compiles tailored checklist of certificates and proofs based on profession and income sources. Places TIN Registration at top if no TIN. | `get_document_checklist` |
| **4** | **Compliance & Audit Reviewer** | Cross-audits all outputs, catches all 6 edge cases, assigns `✅ Passed` or `⚠️ Flagged`, and formats final report with Bangla summary. | `validate_compliance_and_edge_cases` |

---

## 🔄 Architecture & Execution Pipeline
```
User Profile ──▶ [Agent 1: Classifier]  ──(threshold, min_tax)──▶ [Agent 2: Slab Calculator]
                       │                                                    │
                       ▼                                                    ▼
            [Agent 3: Document Organizer]                           (tax computation)
                       │                                                    │
                       └──────────────────▶ [Agent 4: Reviewer] ◀──────────┘
                                                    │
                                                    ▼
                                  [English Markdown Table + Bangla Summary]
```

---

## ✅ Assignment Requirements Covered
| # | Requirement | Status |
|---|---|---|
| 1 | 4 Agents with role, goal, backstory | ✅ |
| 2 | Sequential tasks with `context=[...]` | ✅ |
| 3 | Custom `@tool` (at least 1, we have 4) | ✅ |
| 4 | `output_pydantic` on at least 1 task | ✅ |
| 5 | Final Markdown Table + Bangla Summary | ✅ |
| 6 | All 6 edge cases handled | ✅ |
| 7 | Fallback DB with `try/except` + log | ✅ |
| 8 | `verbose=True` on all agents | ✅ |

## 📦 Step 1: Install Dependencies
Run the cell below to install CrewAI and Google GenAI libraries in the Colab runtime.

In [ ]:
# Install required dependencies
!pip install -q crewai crewai-tools pydantic python-dotenv google-genai

## 🔑 Step 2: Configure API Key
You can configure your **Google Gemini API Key** or **Groq / OpenAI API Key** below.
- **Option A (Recommended):** Add your key to Colab Secrets as `GEMINI_API_KEY`.
- **Option B:** Enter it manually when prompted.

In [ ]:
import os
import getpass

# Try retrieving from Google Colab userdata secrets
try:
    from google.colab import userdata
    api_key = userdata.get('GEMINI_API_KEY') or userdata.get('GOOGLE_API_KEY')
    if api_key:
        os.environ['GEMINI_API_KEY'] = api_key
except Exception:
    pass

if not os.environ.get('GEMINI_API_KEY') and not os.environ.get('GROQ_API_KEY'):
    print('Please enter your Gemini API Key (or set it in Colab Secrets):')
    key = getpass.getpass('GEMINI_API_KEY: ')
    if key:
        os.environ['GEMINI_API_KEY'] = key.strip()

# Sanitize litellm to prevent provider compatibility issues
try:
    import litellm
    litellm.drop_params = True
    _orig_completion = litellm.completion
    def _sanitized_completion(*args, **kwargs):
        if 'messages' in kwargs and isinstance(kwargs['messages'], list):
            for msg in kwargs['messages']:
                if isinstance(msg, dict):
                    msg.pop('cache_breakpoint', None)
        return _orig_completion(*args, **kwargs)
    litellm.completion = _sanitized_completion
except ImportError:
    pass

print('✅ API Key successfully configured!')

## 🗄️ Step 3: Local Fallback Database (Source of Truth)
This database contains the simulated 2026 tax year parameters from the assignment specification.

**Requirement 7**: If remote scraping/API fails, the system logs a `[FALLBACK TRIGGERED]` message and loads this Local DB.

In [ ]:
import json
import os
from pathlib import Path

TAX_DB_DATA = {
    "tax_year": "2026-27",
    "filing_deadline": "2026-11-30",
    "thresholds": {
        "general": 350000,
        "female": 400000,
        "senior_65_plus": 400000,
        "disabled": 475000,
        "freedom_fighter": 500000
    },
    "slabs_after_threshold": [
        {"limit": 100000, "rate": 0.05},
        {"limit": 400000, "rate": 0.10},
        {"limit": 500000, "rate": 0.15},
        {"limit": 500000, "rate": 0.20},
        {"limit": None, "rate": 0.25}
    ],
    "minimum_tax": {
        "dhaka_chattogram_city": 5000,
        "other_city_corporation": 4000,
        "rest_of_country": 3000
    },
    "investment_rebate": {
        "rate": 0.15,
        "max_investment_percent_of_income": 0.20
    },
    "required_docs": {
        "common": ["TIN Certificate", "NID Copy", "Bank Statement"],
        "salaried": ["Salary Certificate", "Employer Details"],
        "business": ["Trade License", "Profit & Loss Statement", "Balance Sheet"],
        "freelancer": ["Foreign Remittance Proof", "Client Invoices"],
        "landlord": ["Rent Agreement", "Municipal Tax Receipt"],
        "investor": ["Investment Certificates (DPS/Sanchaypatra)", "Bank Interest Certificate"]
    }
}

# Save to data/tax_db.json
os.makedirs('data', exist_ok=True)
with open('data/tax_db.json', 'w', encoding='utf-8') as f:
    json.dump(TAX_DB_DATA, f, indent=2)

print('✅ Local Fallback Database created at data/tax_db.json!')

## 🧩 Step 4: Database Loader with Network Fallback (Requirement 7)
Demonstrates the graceful network fallback pattern:
- Attempts to fetch from `https://nbr.gov.bd/api/v1/tax_rules_2026.json`
- On failure, logs `[FALLBACK TRIGGERED]` and uses local `data/tax_db.json`

Also includes all helper functions: taxpayer category resolution, threshold lookup, minimum tax, slab calculation, and document checklist generation.

In [ ]:
import logging
import urllib.request
import urllib.error
from typing import Any, Dict, List, Optional, Tuple

logger = logging.getLogger('AmarTax')
logging.basicConfig(level=logging.INFO, format='%(asctime)s [%(levelname)s] %(name)s: %(message)s')

LOCAL_DB_PATH = Path('data/tax_db.json')
REMOTE_API_URL = 'https://nbr.gov.bd/api/v1/tax_rules_2026.json'


def load_tax_database(source_url: Optional[str] = None) -> Dict[str, Any]:
    """
    Requirement 7: Try remote fetch, except fallback to local DB with log message.
    Attempts to query the remote NBR API. On failure, gracefully falls back to
    the Local Fallback Database with a [FALLBACK TRIGGERED] warning.
    """
    target_url = source_url or REMOTE_API_URL
    logger.info(f'Attempting to query remote tax rules source: {target_url}')
    try:
        req = urllib.request.Request(target_url, headers={'User-Agent': 'AmarTax-AI-Agent/1.0 (NBR Virtual Advisor)'})
        with urllib.request.urlopen(req, timeout=3) as resp:
            if resp.status == 200:
                data = json.loads(resp.read().decode('utf-8'))
                logger.info('Successfully fetched live tax database from remote source.')
                return data
            else:
                raise urllib.error.HTTPError(target_url, resp.status, f'HTTP Error {resp.status}', resp.headers, None)
    except Exception as e:
        logger.warning(
            f'[FALLBACK TRIGGERED] Remote Tax API/Scraping failed: {e}. '
            f"Gracefully falling back to Local Fallback Database at '{LOCAL_DB_PATH}'."
        )

    if not LOCAL_DB_PATH.exists():
        raise FileNotFoundError(f'Local Fallback Database not found at {LOCAL_DB_PATH}')
    with open(LOCAL_DB_PATH, 'r', encoding='utf-8') as f:
        db_data = json.load(f)
        logger.info('Loaded Local Fallback Database successfully as Source of Truth.')
        return db_data


def resolve_taxpayer_category(age: int, gender: str, is_disabled: bool = False, is_freedom_fighter: bool = False) -> Tuple[str, str]:
    """Resolves taxpayer category key and display name based on profile."""
    if is_freedom_fighter:
        return 'freedom_fighter', 'Freedom Fighter'
    if is_disabled:
        return 'disabled', 'Disabled'
    if age >= 65:
        return 'senior_65_plus', 'Senior 65+'
    if gender.strip().lower() in ['female', 'woman', 'f']:
        return 'female', 'Female'
    return 'general', 'General'


def get_tax_threshold(category_key: str, db: Optional[Dict[str, Any]] = None) -> float:
    """Returns the tax-free threshold for a given category."""
    database = db or load_tax_database()
    thresholds = database.get('thresholds', {})
    return float(thresholds.get(category_key, thresholds.get('general', 350000)))


def resolve_minimum_tax(location: str, db: Optional[Dict[str, Any]] = None) -> Tuple[float, str]:
    """Resolves minimum tax amount and region name based on location string."""
    database = db or load_tax_database()
    min_taxes = database.get('minimum_tax', {})
    loc = location.strip().lower()
    if any(c in loc for c in ['dhaka', 'chattogram', 'chittagong']):
        return float(min_taxes.get('dhaka_chattogram_city', 5000)), 'Dhaka/Chattogram City Corporation'
    other_cities = ['gazipur', 'narayanganj', 'rajshahi', 'khulna', 'sylhet',
                    'barishal', 'barisal', 'rangpur', 'mymensingh', 'cumilla', 'comilla']
    if any(c in loc for c in other_cities) or 'city corporation' in loc:
        return float(min_taxes.get('other_city_corporation', 4000)), 'Other City Corporation'
    return float(min_taxes.get('rest_of_country', 3000)), 'Rest of the Country'


def calculate_tax_breakdown(income: float, threshold: float, actual_investment: float, minimum_tax: float, db: Optional[Dict[str, Any]] = None) -> Dict[str, Any]:
    """
    Executes precise slab calculation according to the Local DB rules.
    - Taxable Income = Total Income
    - Slabs begin after the tax-free threshold
    - Allowable Investment = min(actual_investment, 20% * income)
    - Rebate = 15% * Allowable Investment
    - Net Tax = max(Gross Tax - Rebate, Minimum Tax) when income > threshold
    """
    database = db or load_tax_database()
    slabs = database.get('slabs_after_threshold', [])
    rebate_rules = database.get('investment_rebate', {'rate': 0.15, 'max_investment_percent_of_income': 0.20})
    rebate_rate = rebate_rules.get('rate', 0.15)
    max_inv_pct = rebate_rules.get('max_investment_percent_of_income', 0.20)

    if income <= threshold:
        allowable_inv = min(max(0.0, actual_investment), max_inv_pct * max(0.0, income))
        return {
            'total_income': income,
            'tax_free_threshold': threshold,
            'taxable_after_threshold': 0.0,
            'slab_breakdown': ['Income is within the tax-free threshold; 0% tax applies.'],
            'gross_tax': 0.0,
            'allowable_investment': allowable_inv,
            'investment_rebate': 0.0,
            'minimum_tax': minimum_tax,
            'net_payable_tax': 0.0,
            'income_above_threshold': False
        }

    taxable_amount = income - threshold
    remaining = taxable_amount
    gross_tax = 0.0
    slab_breakdown = []

    for slab in slabs:
        limit = slab.get('limit')
        rate = slab.get('rate', 0.0)
        pct = int(rate * 100)
        if limit is None:
            if remaining > 0:
                tax = remaining * rate
                gross_tax += tax
                slab_breakdown.append(f'Remaining {remaining:,.0f} BDT @ {pct}% = {tax:,.0f} BDT')
                remaining = 0.0
            break
        else:
            in_slab = min(remaining, float(limit))
            if in_slab > 0:
                tax = in_slab * rate
                gross_tax += tax
                slab_breakdown.append(f'Next {in_slab:,.0f} BDT @ {pct}% = {tax:,.0f} BDT')
                remaining -= in_slab
            if remaining <= 0:
                break

    allowable_inv = min(max(0.0, actual_investment), max_inv_pct * income)
    rebate = allowable_inv * rebate_rate
    tax_after_rebate = max(0.0, gross_tax - rebate)
    net_payable_tax = max(tax_after_rebate, minimum_tax)

    return {
        'total_income': income,
        'tax_free_threshold': threshold,
        'taxable_after_threshold': taxable_amount,
        'slab_breakdown': slab_breakdown,
        'gross_tax': gross_tax,
        'allowable_investment': allowable_inv,
        'investment_rebate': rebate,
        'minimum_tax': minimum_tax,
        'net_payable_tax': net_payable_tax,
        'income_above_threshold': True
    }


def get_required_documents(profession: str, income_sources: Optional[List[str]] = None, has_tin: bool = True, has_investment: bool = False, db: Optional[Dict[str, Any]] = None) -> Tuple[List[str], bool]:
    """Generates tailored checklist based on profession, income sources, and TIN status."""
    database = db or load_tax_database()
    req_docs = database.get('required_docs', {})
    common_docs = list(req_docs.get('common', ['TIN Certificate', 'NID Copy', 'Bank Statement']))
    checklist = []

    # Edge Case 5: Missing TIN -> TIN Registration at top
    if not has_tin:
        checklist.append('TIN Registration')
        common_docs = [d for d in common_docs if d != 'TIN Certificate']

    prof_clean = profession.strip().lower()
    sources = [s.strip().lower() for s in (income_sources or [])]
    doc_keys_found = []

    for key in ['salaried', 'business', 'freelancer', 'landlord', 'investor']:
        if key in prof_clean or any(key in s for s in sources):
            doc_keys_found.append(key)

    # Keyword mappings for professions
    if any(k in prof_clean or any(k in s for s in sources) for k in ['engineer', 'officer', 'manager', 'employee', 'teacher', 'salary']):
        if 'salaried' not in doc_keys_found:
            doc_keys_found.append('salaried')
    if any(k in prof_clean or any(k in s for s in sources) for k in ['freelance', 'contractor', 'remote', 'remittance']):
        if 'freelancer' not in doc_keys_found:
            doc_keys_found.append('freelancer')
    if any(k in prof_clean or any(k in s for s in sources) for k in ['rent', 'landlord', 'property', 'house owner']):
        if 'landlord' not in doc_keys_found:
            doc_keys_found.append('landlord')
    if has_investment or any(k in prof_clean or any(k in s for s in sources) for k in ['invest', 'dps', 'sanchaypatra', 'shares', 'stocks']):
        if 'investor' not in doc_keys_found:
            doc_keys_found.append('investor')

    is_general_fallback = False
    if not doc_keys_found:
        # Edge Case 6: Profession not in DB -> Fall back to general checklist
        is_general_fallback = True
        checklist.extend(common_docs)
    else:
        checklist.extend(common_docs)
        for key in doc_keys_found:
            for item in req_docs.get(key, []):
                if item not in checklist:
                    checklist.append(item)

    # Deduplicate while preserving order
    deduped = []
    for item in checklist:
        if item not in deduped:
            deduped.append(item)
    return deduped, is_general_fallback

## 📐 Step 5: Pydantic Structured Output Models (Requirement 4)
Use `output_pydantic` in at least one Task to enforce structured, validated outputs.

In [ ]:
from pydantic import BaseModel, Field


class TaxpayerEligibilityOutput(BaseModel):
    """Structured output from Agent 1 (Classifier / Senior Taxpayer Classification Officer)."""
    taxpayer_category: str = Field(..., description='Category: Female, General, Senior 65+, Disabled, Freedom Fighter')
    is_mandatory: bool = Field(..., description='Whether filing an income tax return is mandatory')
    tax_free_threshold: float = Field(..., description='Tax-free threshold amount in BDT')
    minimum_tax: float = Field(..., description='Minimum tax payable based on location in BDT')
    location_tier: str = Field(..., description='Classification of taxpayer\'s location')
    eligibility_notes: str = Field(..., description='Explanation of mandatory filing rule and age restrictions')
    flags: List[str] = Field(default_factory=list, description='Any detected edge case warnings or flags')


class TaxCalculationOutput(BaseModel):
    """Structured output from Agent 2 (Slab Calculator / Chartered Tax Accountant)."""
    total_income: float = Field(..., description='Total annual income in BDT')
    tax_free_threshold: float = Field(..., description='Applicable tax-free threshold in BDT')
    taxable_income_after_threshold: float = Field(..., description='Income amount subject to tax slabs in BDT')
    gross_tax: float = Field(..., description='Gross tax before rebate in BDT')
    slab_breakdown: List[str] = Field(default_factory=list, description='Step-by-step breakdown across tax slabs')
    allowable_investment: float = Field(..., description='Allowable investment amount (capped at 20% of income)')
    investment_rebate: float = Field(..., description='Investment rebate amount (15% of allowable investment)')
    minimum_tax: float = Field(..., description='Minimum applicable tax for taxpayer\'s location')
    net_payable_tax: float = Field(..., description='Net tax payable after rebate comparison against minimum tax')
    calculation_summary: str = Field(..., description='Summary of computation logic and constraints')
    flags: List[str] = Field(default_factory=list, description='Any computational flags')


class DocumentChecklistOutput(BaseModel):
    """Structured output from Agent 3 (Document Organizer / Return Filing Documentation Officer)."""
    profession: str = Field(..., description='Taxpayer profession considered')
    has_tin: bool = Field(..., description='Whether the taxpayer has a TIN')
    documents_needed: List[str] = Field(default_factory=list, description='List of required documents and proofs')
    is_fallback_checklist: bool = Field(False, description='True if profession was not in DB and general list was used')
    notes: str = Field('', description='Additional filing tips or guidelines')


class TaxReadinessReport(BaseModel):
    """Final structured output from Agent 4 (Compliance Reviewer / Validator Agent)."""
    taxpayer_category: str = Field(..., description='Taxpayer category, e.g. Female, General')
    total_income: float = Field(..., description='Total annual income in BDT')
    tax_free_threshold: float = Field(..., description='Tax-free threshold in BDT')
    gross_tax: float = Field(..., description='Gross slab-based tax in BDT')
    investment_rebate: float = Field(..., description='Investment rebate in BDT')
    minimum_tax: float = Field(..., description='Location minimum tax in BDT')
    net_payable_tax: float = Field(..., description='Net payable tax in BDT')
    documents_needed: List[str] = Field(default_factory=list, description='All required documents')
    compliance_status: str = Field(..., description="Final audit status: '✅ Passed' or '⚠️ Flagged'")
    identified_issues: List[str] = Field(default_factory=list, description='All caught errors, flags, or edge cases')
    markdown_table: str = Field(..., description='English Markdown Table formatted strictly as required')
    bangla_summary: str = Field(..., description='3-4 line Bangla summary of the tax readiness report')

## 🛠️ Step 6: Custom Tools — `@tool` (Requirement 3)
We build **4 custom tools** using CrewAI's `@tool` decorator:
1. `lookup_tax_db` — query the tax database
2. `calculate_slab_tax` — compute slab tax, rebate, net payable
3. `get_document_checklist` — generate tailored document checklist
4. `validate_compliance_and_edge_cases` — catch all 6 edge cases

In [ ]:
from crewai.tools import tool


@tool('lookup_tax_db')
def lookup_tax_db(query_type: str, category: str = '', location: str = '') -> str:
    """
    Look up tax rules, thresholds, minimum taxes, slabs, and rebate parameters from the tax database.
    Demonstrates network fallback mechanism (try remote API/scraper, except fall back to local DB).

    Args:
        query_type: One of 'thresholds', 'minimum_tax', 'slabs', 'rebate_rules', 'all'
        category: Optional category name ('female', 'general', 'senior_65_plus', 'disabled', 'freedom_fighter')
        location: Optional taxpayer location ('Dhaka', 'Chattogram', 'Rangpur', etc.)
    """
    db = load_tax_database()
    query = query_type.strip().lower()

    if query == 'thresholds':
        if category:
            cat_key = category.strip().lower().replace(' ', '_')
            threshold = get_tax_threshold(cat_key, db)
            return json.dumps({
                'category': category, 'category_key': cat_key,
                'tax_free_threshold_bdt': threshold, 'tax_year': db.get('tax_year', '2026-27')
            }, indent=2)
        return json.dumps({
            'tax_year': db.get('tax_year', '2026-27'),
            'filing_deadline': db.get('filing_deadline', '2026-11-30'),
            'thresholds': db.get('thresholds', {})
        }, indent=2)

    if query == 'minimum_tax':
        min_tax, tier = resolve_minimum_tax(location, db)
        return json.dumps({
            'location_provided': location, 'matched_tier': tier,
            'minimum_tax_bdt': min_tax, 'all_minimum_taxes': db.get('minimum_tax', {})
        }, indent=2)

    if query == 'slabs':
        return json.dumps({'slabs_after_threshold': db.get('slabs_after_threshold', [])}, indent=2)

    if query == 'rebate_rules':
        return json.dumps({'investment_rebate': db.get('investment_rebate', {})}, indent=2)

    return json.dumps(db, indent=2)


@tool('calculate_slab_tax')
def calculate_slab_tax(income: float, threshold: float, investment: float, minimum_tax: float) -> str:
    """
    Accurately calculates slab tax, investment rebate, and net payable tax
    strictly according to the Local Fallback Database rules. Never invents rates.

    Args:
        income: Total annual taxable income in BDT.
        threshold: Tax-free threshold in BDT for taxpayer's category.
        investment: Total eligible investment amount in BDT.
        minimum_tax: Location-based minimum tax in BDT.
    """
    db = load_tax_database()
    res = calculate_tax_breakdown(float(income), float(threshold), float(investment), float(minimum_tax), db)
    return json.dumps(res, indent=2)


@tool('get_document_checklist')
def get_document_checklist(profession: str, income_sources_csv: str = '', has_tin: bool = True, has_investment: bool = False) -> str:
    """
    Generates customized document checklist based on profession, income sources, and TIN status.
    Handles edge cases:
      - If no TIN: adds 'TIN Registration' at the top of the checklist.
      - If profession not in DB: gracefully falls back to the general checklist.

    Args:
        profession: Taxpayer profession (e.g., 'Software Engineer', 'Doctor', 'Salaried')
        income_sources_csv: Comma-separated income sources (e.g., 'salary, freelancing')
        has_tin: Boolean indicating whether taxpayer already possesses a TIN
        has_investment: Boolean indicating whether taxpayer has investments to claim rebate
    """
    db = load_tax_database()
    sources = [s.strip() for s in income_sources_csv.split(',') if s.strip()]
    checklist, is_fallback = get_required_documents(profession, sources, has_tin, has_investment, db)
    result = {
        'profession': profession, 'income_sources': sources, 'has_tin': has_tin,
        'is_general_fallback': is_fallback, 'documents_needed': checklist,
        'notes': ('Profession not recognized in DB; applied general fallback checklist.'
                  if is_fallback else 'Customized checklist based on declared profession and sources.')
    }
    return json.dumps(result, indent=2)


@tool('validate_compliance_and_edge_cases')
def validate_compliance_and_edge_cases(
    age: int, income: float, investment: float, threshold: float,
    claims_required_to_file: bool = False, has_tin: bool = True,
    wants_independent_filing: bool = True, profession_is_fallback: bool = False
) -> str:
    """
    Cross-checks outputs and catches all required assignment edge cases:
      1. Age < 18 and file independently -> ⚠️ Minor cannot file independently
      2. Negative or impossible income -> Flag as invalid input
      3. Investment > total income -> ⚠️ Investment exceeds income, please verify
      4. Income below threshold but claims 'required to file' -> Inform filing not mandatory
      5. No TIN -> 'TIN Registration' must be added at top of checklist
      6. Profession not in DB -> Fall back to general checklist

    Returns compliance audit status ('✅ Passed' or '⚠️ Flagged') and list of flags.
    """
    flags = []

    # Edge Case 1: Minor independent filing
    if age < 18 and wants_independent_filing:
        flags.append('⚠️ Minor cannot file independently')

    # Edge Case 2: Negative or impossible income
    if income < 0:
        flags.append('⚠️ Flag as invalid input: Income cannot be negative')

    # Edge Case 3: Investment exceeds income
    if investment > income and income > 0:
        flags.append('⚠️ Investment exceeds income, please verify')

    # Edge Case 4: Income below threshold but claims required to file
    if income <= threshold and claims_required_to_file:
        flags.append(
            f'⚠️ Inform them that filing is not mandatory (income {income:,.0f} BDT is within '
            f'the tax-free threshold of {threshold:,.0f} BDT, but check the Minimum Tax rule)'
        )

    # Edge Case 5: No TIN
    if not has_tin:
        flags.append("⚠️ No TIN: 'TIN Registration' must be added at the top of the checklist")

    # Edge Case 6: Profession not in DB
    if profession_is_fallback:
        flags.append('⚠️ Profession not in DB: Fall back to the general checklist')

    status = '⚠️ Flagged' if flags else '✅ Passed'
    return json.dumps({'compliance_status': status, 'total_flags': len(flags), 'flags': flags}, indent=2)

## 🤖 Step 7: Define The 4 Specialized Agents (Requirements 1 & 8)
- Every agent has a **role**, **goal**, and **backstory**.
- All agents use `verbose=True`.

In [ ]:
from crewai import Agent, LLM

# Initialize CrewAI LLM with Gemini 3.5 Flash Lite
gemini_key = os.environ.get('GEMINI_API_KEY') or os.environ.get('GOOGLE_API_KEY')
llm = LLM(model='gemini/gemini-3.5-flash-lite', api_key=gemini_key, temperature=0.1)

classifier_agent = Agent(
    role='Senior Taxpayer Classification Officer',
    goal=(
        'Based on the user\'s profile (Age, Gender, Profession, Income, Investment, Location, TIN status), '
        'determine: 1) Whether filing a return is mandatory, 2) Taxpayer category (General / Female / '
        'Senior 65+ / Disabled / Freedom Fighter), 3) The tax-free threshold for that category, and '
        '4) The minimum tax based on location. Strictly flag if anyone under 18 attempts to file independently.'
    ),
    backstory=(
        'You are a Senior Taxpayer Classification Officer with over 15 years of distinguished service '
        'at the National Board of Revenue (NBR) in Bangladesh. You are an authority on individual taxpayer '
        'eligibility criteria, statutory thresholds, and regional minimum tax jurisdictions. You are '
        'rigorous and detail-oriented: you strictly apply NBR definitions and legal age requirements. Under '
        'no circumstances may a minor under 18 file an independent return, and any such attempt must be flagged.'
    ),
    tools=[lookup_tax_db],
    llm=llm,
    verbose=True,
    memory=False
)

calculator_agent = Agent(
    role='Chartered Tax Accountant',
    goal=(
        'Using the Classifier\'s output as context, accurately calculate: 1) Total tax according to slabs, '
        '2) Allowable investment and investment rebate (15% of allowable investment capped at 20% of income), '
        'and 3) Net payable tax (compared against minimum tax). Never invent rates; strictly use the slab '
        'and rebate rules from the Local Database.'
    ),
    backstory=(
        'You are a Fellow Chartered Accountant (FCA) and premier tax computational specialist in Bangladesh. '
        'You possess an exacting command over the Bangladesh Income Tax Act rules, progressive tax slabs, '
        'and statutory investment tax credits. You treat the local database as the immutable '
        'source of truth. You apply thresholds cleanly, calculate slab increments step-by-step, verify allowable '
        'investment caps (min(actual, 20% of income)), compute the 15% rebate, and ensure the taxpayer pays '
        'at least the statutory location minimum tax if income exceeds the threshold.'
    ),
    tools=[lookup_tax_db, calculate_slab_tax],
    llm=llm,
    verbose=True,
    memory=False
)

organizer_agent = Agent(
    role='Return Filing Documentation Officer',
    goal=(
        'Generate an exhaustive, customized checklist of required documents based on the taxpayer\'s '
        'profession and income sources (e.g., Salaried -> Salary Certificate, Freelancer -> Remittance Proof, '
        'Landlord -> Rent Agreement, Investor -> Investment Certificates). If the user does not have a TIN, '
        'ensure \'TIN Registration\' is placed at the top. If profession is not in the database, fall back to '
        'the general checklist.'
    ),
    backstory=(
        'You are an experienced Return Filing Documentation Officer at the NBR Central Service Center. '
        'You have guided thousands of taxpayers through the compliance documentation process. You understand '
        'precisely which papers, certificates, deduction proofs, and bank statements are legally necessary for '
        'salaried individuals, freelancers, business proprietors, landlords, and investors. You ensure no '
        'taxpayer faces audit penalties due to missing documentation.'
    ),
    tools=[get_document_checklist],
    llm=llm,
    verbose=True,
    memory=False
)

reviewer_agent = Agent(
    role='Compliance & Audit Reviewer',
    goal=(
        'Cross-check and audit the outputs of the Classifier, Calculator, and Document Organizer. '
        'Detect and flag all edge cases: 1) Minor filing independently, 2) Negative or impossible income, '
        '3) Investment exceeding total income, 4) Income below threshold but claims required to file, '
        '5) Missing TIN without registration, 6) Unlisted profession fallback. Assign a final audit '
        'status of \'✅ Passed\' or \'⚠️ Flagged\'. Deliver the final Tax Readiness Report consisting of an '
        'English Markdown Table and a concise 3-4 line Bangla summary.'
    ),
    backstory=(
        'You are the Chief Compliance & Audit Reviewer at the National Board of Revenue. You possess an '
        'unforgiving eye for calculation anomalies, contradictory declarations, and statutory oversights. '
        'You evaluate whether the taxpayer\'s assessment is fully validated or requires urgent remediation. '
        'You synthesize the complex multi-agent outputs into an executive-ready English Markdown Table and '
        'an empathetic, fluent 3-4 line Bangla summary for the citizen.'
    ),
    tools=[validate_compliance_and_edge_cases],
    llm=llm,
    verbose=True,
    memory=False
)

print('✅ All 4 agents created successfully!')

## 📋 Step 8: Define Tasks & Context Passing (Requirement 2)
Sequential pipeline: **Classifier → Calculator → Organizer → Reviewer**

Each task passes its output as `context=[...]` to the next agent.

In [ ]:
from crewai import Task, Crew, Process


def build_tax_crew(profile: dict) -> Crew:
    """Builds and configures the Amar Tax CrewAI system for a given user profile."""

    task1 = Task(
        description=(
            f"Analyze the following taxpayer profile:\n"
            f"  - Age: {profile.get('age')}\n"
            f"  - Gender: {profile.get('gender')}\n"
            f"  - Profession: {profile.get('profession')}\n"
            f"  - Total Annual Income: {profile.get('total_income', 0):,} BDT\n"
            f"  - Investment: {profile.get('investment', 0):,} BDT\n"
            f"  - Location: {profile.get('location')}\n"
            f"  - Has TIN: {profile.get('has_tin', True)}\n"
            f"  - Claims Required to File: {profile.get('claims_required_to_file', False)}\n"
            f"  - Wants Independent Filing: {profile.get('wants_independent_filing', True)}\n"
            f"  - Disabled: {profile.get('is_disabled', False)}\n"
            f"  - Freedom Fighter: {profile.get('is_freedom_fighter', False)}\n\n"
            f"Instructions:\n"
            f"1. Use `lookup_tax_db` to determine the taxpayer category and its corresponding tax-free threshold.\n"
            f"2. Determine the location-based minimum tax (Dhaka/Chattogram City: 5,000, Other City Corp: 4,000, Rest: 3,000).\n"
            f"3. Check whether return filing is mandatory.\n"
            f"4. CRITICAL RULE: If the taxpayer is under 18 years old and wants to file independently, you MUST flag:\n"
            f"   '⚠️ Minor cannot file independently'.\n"
            f"5. If income is below the tax-free threshold, note that filing is not mandatory."
        ),
        expected_output='Structured TaxpayerEligibilityOutput with category, threshold, minimum tax, notes, flags.',
        agent=classifier_agent,
        output_pydantic=TaxpayerEligibilityOutput
    )

    task2 = Task(
        description=(
            f"Using the classification details from the Senior Taxpayer Classification Officer as context, "
            f"compute the complete tax obligation for the taxpayer with total income {profile.get('total_income', 0):,} BDT "
            f"and investment {profile.get('investment', 0):,} BDT.\n\n"
            f"Instructions:\n"
            f"1. Use `calculate_slab_tax` and `lookup_tax_db` tools to compute the figures strictly according to Local DB rules.\n"
            f"2. Slabs begin after the tax-free threshold determined in Task 1.\n"
            f"3. Slabs after threshold: First 100,000 @ 5%, Next 400,000 @ 10%, Next 500,000 @ 15%, Next 500,000 @ 20%, Remaining @ 25%.\n"
            f"4. Allowable Investment = min(actual_investment, 20% * income).\n"
            f"5. Investment Rebate = 15% * Allowable Investment.\n"
            f"6. Net Tax = max(Gross Tax - Rebate, Minimum Tax) when income is above the threshold. If income <= threshold, gross=0, net=0.\n"
            f"7. Flag if investment > income or if income is negative."
        ),
        expected_output='Structured TaxCalculationOutput with gross tax, rebate, minimum tax, and net payable tax.',
        agent=calculator_agent,
        context=[task1],
        output_pydantic=TaxCalculationOutput
    )

    sources_str = ', '.join(profile.get('income_sources', []))
    task3 = Task(
        description=(
            f"Generate the comprehensive document checklist for this taxpayer:\n"
            f"  - Profession: {profile.get('profession')}\n"
            f"  - Income Sources: {sources_str}\n"
            f"  - Has TIN: {profile.get('has_tin', True)}\n"
            f"  - Total Investment: {profile.get('investment', 0):,} BDT\n\n"
            f"Instructions:\n"
            f"1. Use `get_document_checklist` to query the required documents.\n"
            f"2. CRITICAL RULE: If the user does not have a TIN (has_tin=False), 'TIN Registration' MUST be added at the top.\n"
            f"3. If the profession is not in the database, fall back to the general checklist.\n"
            f"4. If taxpayer has investments (DPS, Sanchaypatra, etc.), include investment proof certificates."
        ),
        expected_output='Structured DocumentChecklistOutput with required documents list.',
        agent=organizer_agent,
        context=[task1],
        output_pydantic=DocumentChecklistOutput
    )

    task4 = Task(
        description=(
            f"Audit and cross-check the complete findings from the Senior Taxpayer Classification Officer, "
            f"Chartered Tax Accountant, and Return Filing Documentation Officer.\n\n"
            f"User Profile Summary: Age={profile.get('age')}, Gender={profile.get('gender')}, "
            f"Profession={profile.get('profession')}, Income={profile.get('total_income', 0):,} BDT, "
            f"Investment={profile.get('investment', 0):,} BDT, Location={profile.get('location')}, "
            f"Has TIN={profile.get('has_tin', True)}, Claims Required={profile.get('claims_required_to_file', False)}, "
            f"Wants Independent Filing={profile.get('wants_independent_filing', True)}.\n\n"
            f"Instructions:\n"
            f"1. Use `validate_compliance_and_edge_cases` to verify all 6 edge cases.\n"
            f"2. Assign final compliance status: '✅ Passed' or '⚠️ Flagged'.\n"
            f"3. Generate the Final Tax Readiness Report containing:\n"
            f"   A) English Markdown Table with the exact fields:\n"
            f"      | Field | Result |\n"
            f"      | :--- | :--- |\n"
            f"      | Taxpayer Category | <Category> |\n"
            f"      | Total Income | <Income> BDT |\n"
            f"      | Tax-free Threshold | <Threshold> BDT |\n"
            f"      | Gross Tax (Slab-based) | <Gross Tax> BDT |\n"
            f"      | Investment Rebate (15%) | <Rebate> BDT |\n"
            f"      | Minimum Tax (<Location>) | <Min Tax> BDT |\n"
            f"      | Net Payable Tax | <Net Tax> BDT |\n"
            f"      | Documents Needed | <Comma-separated list> |\n"
            f"      | Compliance Status | ✅ Passed / ⚠️ Flagged |\n\n"
            f"   B) Warning / Flag Section (if any flags are raised).\n"
            f"   C) Bangla Summary (strictly 3 to 4 lines explaining total income, tax-free threshold, "
            f"      net payable tax, and filing advice in clear, natural Bengali)."
        ),
        expected_output='The complete Tax Readiness Report containing the English Markdown Table, any compliance edge case flags, and the 3-4 line Bangla summary.',
        agent=reviewer_agent,
        context=[task1, task2, task3]
    )

    return Crew(
        agents=[classifier_agent, calculator_agent, organizer_agent, reviewer_agent],
        tasks=[task1, task2, task3, task4],
        process=Process.sequential,
        verbose=True
    )


def run_tax_advisory(profile: dict):
    """Executes the Amar Tax advisory crew on a taxpayer profile."""
    crew = build_tax_crew(profile)
    return str(crew.kickoff())

---
# 🧪 Test Cases
The following cells execute all test scenarios from the assignment, including the **Example Scenario** and all **6 Edge Cases**.

## 🧪 Test Case 1: Example Scenario (Baseline Acceptance)
**Input**:
> *"I am a 34-year-old woman working as a software engineer, and I live in Dhaka. My annual salary is 1,200,000 BDT and I earn 300,000 BDT from freelancing (remittance comes through the bank). I have invested 200,000 BDT in DPS and Sanchaypatra. I have a TIN."*

**Expected Output**:
- **Category**: Female → Threshold = 400,000 BDT
- **Gross Tax**: 140,000 BDT (slab-based on 1,100,000 BDT taxable)
- **Allowable Investment**: min(200,000, 20% × 1,500,000) = 200,000 BDT
- **Rebate**: 15% × 200,000 = 30,000 BDT
- **Net Tax** = max(140,000 − 30,000, 5,000) = **110,000 BDT**
- **Compliance Status** = **✅ Passed**

In [ ]:
from IPython.display import display, Markdown

tc1_profile = {
    'age': 34,
    'gender': 'Female',
    'profession': 'Software Engineer',
    'total_income': 1500000.0,
    'investment': 200000.0,
    'location': 'Dhaka',
    'has_tin': True,
    'wants_independent_filing': True,
    'claims_required_to_file': False,
    'income_sources': ['salaried', 'freelancer', 'investor'],
    'is_disabled': False,
    'is_freedom_fighter': False,
    'raw_prompt': (
        'I am a 34-year-old woman working as a software engineer, and I live in Dhaka. '
        'My annual salary is 1,200,000 BDT and I earn 300,000 BDT from freelancing '
        '(remittance comes through the bank). I have invested 200,000 BDT in DPS and '
        'Sanchaypatra. I have a TIN.'
    )
}

report1 = run_tax_advisory(tc1_profile)
display(Markdown(report1))

## 🧪 Test Case 2: Minor Independent Filing (Edge Case 1)
**Input**: 16-year-old male, Income 500,000 BDT, wants to file independently.

**Expected Behavior**: `⚠️ Minor cannot file independently` flag raised.

In [ ]:
tc2_profile = {
    'age': 16,
    'gender': 'Male',
    'profession': 'Freelancer',
    'total_income': 500000.0,
    'investment': 0.0,
    'location': 'Dhaka',
    'has_tin': True,
    'wants_independent_filing': True,
    'claims_required_to_file': False,
    'income_sources': ['freelancer'],
    'is_disabled': False,
    'is_freedom_fighter': False,
    'raw_prompt': 'I am 16 years old earning 500,000 BDT from online freelancing and want to file my taxes independently.'
}

report2 = run_tax_advisory(tc2_profile)
display(Markdown(report2))

## 🧪 Test Case 3: Below Tax-Free Threshold in Rangpur (Edge Case 4)
**Input**: 45-year-old male teacher, Income 300,000 BDT in Rangpur, claims required to file.

**Expected Behavior**: Filing not mandatory (income 300,000 < threshold 350,000), with an explanatory warning message.

In [ ]:
tc3_profile = {
    'age': 45,
    'gender': 'Male',
    'profession': 'Teacher',
    'total_income': 300000.0,
    'investment': 0.0,
    'location': 'Rangpur',
    'has_tin': True,
    'wants_independent_filing': True,
    'claims_required_to_file': True,
    'income_sources': ['salaried'],
    'is_disabled': False,
    'is_freedom_fighter': False,
    'raw_prompt': 'I am a 45-year-old male teacher living in Rangpur earning 300,000 BDT annually. I believe I am required to file.'
}

report3 = run_tax_advisory(tc3_profile)
display(Markdown(report3))

## 🧪 Test Case 4: Excessive Investment (Edge Case 3)
**Input**: 40-year-old male businessman, Income 800,000 BDT, Investment 1,000,000 BDT.

**Expected Behavior**: `⚠️ Investment exceeds income, please verify` flag raised.

In [ ]:
tc4_profile = {
    'age': 40,
    'gender': 'Male',
    'profession': 'Business',
    'total_income': 800000.0,
    'investment': 1000000.0,
    'location': 'Chattogram',
    'has_tin': True,
    'wants_independent_filing': True,
    'claims_required_to_file': False,
    'income_sources': ['business', 'investor'],
    'is_disabled': False,
    'is_freedom_fighter': False,
    'raw_prompt': 'I am a 40-year-old businessman in Chattogram with 800,000 BDT income and 1,000,000 BDT in investments.'
}

report4 = run_tax_advisory(tc4_profile)
display(Markdown(report4))

## 🧪 Test Case 5: Missing TIN (Edge Case 5)
**Input**: 28-year-old male software engineer, Income 600,000 BDT in Dhaka, **no TIN**.

**Expected Behavior**: `⚠️ No TIN` flag and `TIN Registration` must appear at the **top** of the document checklist.

In [ ]:
tc5_profile = {
    'age': 28,
    'gender': 'Male',
    'profession': 'Software Engineer',
    'total_income': 600000.0,
    'investment': 50000.0,
    'location': 'Dhaka',
    'has_tin': False,
    'wants_independent_filing': True,
    'claims_required_to_file': False,
    'income_sources': ['salaried'],
    'is_disabled': False,
    'is_freedom_fighter': False,
    'raw_prompt': 'I am 28 years old, earning 600,000 BDT in Dhaka, but I do NOT have a TIN yet.'
}

report5 = run_tax_advisory(tc5_profile)
display(Markdown(report5))

## 🧪 Test Case 6: Unlisted Profession Fallback (Edge Case 6)
**Input**: 35-year-old male sculptor / visual artist, Income 450,000 BDT in Sylhet.

**Expected Behavior**: `⚠️ Profession not in DB: Fall back to the general checklist` flag raised. Document checklist uses the general fallback list.

In [ ]:
tc6_profile = {
    'age': 35,
    'gender': 'Male',
    'profession': 'Sculptor / Visual Artist',
    'total_income': 450000.0,
    'investment': 0.0,
    'location': 'Sylhet',
    'has_tin': True,
    'wants_independent_filing': True,
    'claims_required_to_file': False,
    'income_sources': [],
    'is_disabled': False,
    'is_freedom_fighter': False,
    'raw_prompt': 'I am a 35-year-old sculptor living in Sylhet earning 450,000 BDT.'
}

report6 = run_tax_advisory(tc6_profile)
display(Markdown(report6))

---

## 📝 Assumptions & Verification Matrix

### Key Assumptions:
1. **Taxable Income = Total Income**: No separate exemptions or deductions are subtracted before applying the tax-free threshold.
2. **Threshold Begins First**: Slabs begin strictly after the taxpayer's threshold (General: 350k, Female: 400k, Senior 65+: 400k, Disabled: 475k, Freedom Fighter: 500k).
3. **Investment Rebate**: Capped at `min(actual_investment, 20% * income) * 15%`.
4. **Minimum Tax**: Dhaka/Chattogram City (5,000 BDT), Other City Corporations (4,000 BDT), Rest of country (3,000 BDT). Net tax = `max(Gross - Rebate, Minimum Tax)` when income exceeds threshold.
5. **Source of Truth**: The local fallback DB `data/tax_db.json` is treated as ground truth if remote endpoints are unavailable.

### Edge Cases Summary:
| # | Edge Case | Trigger Condition | Expected Flag |
|---|---|---|---|
| 1 | Minor Independent Filing | Age < 18, wants_independent_filing=True | `⚠️ Minor cannot file independently` |
| 2 | Negative/Impossible Income | income < 0 | `⚠️ Flag as invalid input` |
| 3 | Investment > Income | investment > income | `⚠️ Investment exceeds income, please verify` |
| 4 | Below Threshold + Claims Required | income ≤ threshold, claims_required_to_file=True | `⚠️ Inform filing not mandatory` |
| 5 | Missing TIN | has_tin=False | `⚠️ No TIN: TIN Registration at top` |
| 6 | Unlisted Profession | profession not recognized in DB | `⚠️ Profession not in DB: general fallback` |